# 🔱 Damru AI — One-Tap Training (Colab / Kaggle T4)

QLoRA fine-tune of an open base model on **Damaru-ai/damru-train** (the clean,
deduped, decontaminated, domain-balanced dataset produced by `prep_training_data.py`).

**Order:** GPU check → install → config → HF login → train (SFT) → eval → push.

> Runtime → Change runtime type → **GPU (T4)** zaroor select karna.

In [ ]:
# 1) GPU check
!nvidia-smi || echo 'NO GPU! Runtime -> Change runtime type -> GPU'

In [ ]:
# 2) Install deps (QLoRA stack)
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" \
  "bitsandbytes>=0.43" "accelerate>=0.33" "datasets>=2.20" huggingface_hub
print('deps ok')

In [ ]:
# 3) CONFIG -- yahan apne hisaab se badlo
import os

# Base model: general tutor ke liye Instruct, pure coder ke liye Coder.
BASE_MODEL   = 'Qwen/Qwen2.5-3B-Instruct'   # ya 'Qwen/Qwen2.5-Coder-3B-Instruct'
TRAIN_REPO   = 'Damaru-ai/damru-train'      # prep_training_data.py ka output
TRAIN_SPLIT  = 'train'
VAL_SPLIT    = 'validation'
OUT_REPO     = 'Damaru-ai/damru-tutor-lora' # jahan LoRA push hogi

# Training knobs (T4-safe)
MAX_SEQ_LEN  = 2048
BATCH        = 2
GRAD_ACCUM   = 8        # effective batch = 16
LR           = 2e-4
EPOCHS       = 1        # overfit/forgetting se bachne ke liye 1
MAX_STEPS    = -1       # -1 = full epoch; chhota test ke liye e.g. 200
LORA_R       = 16
LORA_ALPHA   = 32
for k in list(globals()):
    if k.isupper():
        os.environ[k] = str(globals()[k])
print('config set ->', BASE_MODEL, '|', TRAIN_REPO, '->', OUT_REPO)

In [ ]:
# 4) HuggingFace login (token Colab Secrets se ya prompt se)
from huggingface_hub import login
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    pass
if not tok:
    import getpass
    tok = getpass.getpass('HF token (write): ')
login(tok)
os.environ['HF_TOKEN'] = tok
print('logged in')

In [ ]:
# 5) Load dataset (ChatML 'text' column expected from prep)
from datasets import load_dataset
train_ds = load_dataset(TRAIN_REPO, split=TRAIN_SPLIT)
try:
    val_ds = load_dataset(TRAIN_REPO, split=VAL_SPLIT)
except Exception:
    val_ds = None
print('train rows:', len(train_ds), '| cols:', train_ds.column_names)

# pick the text field produced by prep (chatml / text / messages)
TEXT_FIELD = next((c for c in ['text','chatml','content'] if c in train_ds.column_names), None)
print('TEXT_FIELD =', TEXT_FIELD)

In [ ]:
# 6) 4-bit base model + LoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             device_map='auto', torch_dtype=torch.float16)
lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'])
print('model + lora ready')

In [ ]:
# 7) Train (SFT). Agar TEXT_FIELD None ho to messages se ChatML banao.
from trl import SFTTrainer, SFTConfig

if TEXT_FIELD is None and 'messages' in train_ds.column_names:
    def _fmt(ex):
        return {'text': tok.apply_chat_template(ex['messages'], tokenize=False)}
    train_ds = train_ds.map(_fmt)
    if val_ds is not None:
        val_ds = val_ds.map(_fmt)
    TEXT_FIELD = 'text'

cfg = SFTConfig(output_dir='damru_out', per_device_train_batch_size=BATCH,
                gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
                num_train_epochs=EPOCHS, max_steps=MAX_STEPS, bf16=False, fp16=True,
                logging_steps=20, save_steps=500, save_total_limit=2,
                max_seq_length=MAX_SEQ_LEN, packing=True, dataset_text_field=TEXT_FIELD,
                warmup_ratio=0.03, lr_scheduler_type='cosine',
                gradient_checkpointing=True, report_to='none')
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds,
                     eval_dataset=val_ds, peft_config=lora)
trainer.train()
print('TRAINING DONE')

In [ ]:
# 8) Push LoRA adapter to HF
trainer.model.push_to_hub(OUT_REPO)
tok.push_to_hub(OUT_REPO)
print('pushed ->', OUT_REPO)

In [ ]:
# 9) Quick smoke test
from transformers import pipeline
msgs = [{'role':'system','content':'You are Damru, a helpful exam + coding tutor.'},
        {'role':'user','content':'Explain the nursing role in managing a patient with hypoglycemia.'}]
prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
pipe = pipeline('text-generation', model=trainer.model, tokenizer=tok)
print(pipe(prompt, max_new_tokens=300, do_sample=True, temperature=0.7)[0]['generated_text'][len(prompt):])

## ✅ Aage
- Coding eval: repo me `eval/coding_eval.py` (HumanEval/MBPP)
- 🩺 Nursing eval: `eval/nursing_eval.py` — `MODEL_ID` set karke chalao (July-3 exam metric)
- LoRA merge + GGUF banakar HF Spaces / phone pe deploy.